# Contoso Events — synthetic badge scans
Import into Fabric, choose a supported **Spark** runtime and attach your **default lakehouse** in the notebook explorer. No workspace or lakehouse identifiers are embedded. Save after attaching; Fabric supplies environment-specific metadata. Run the base CSV loader first. Upload `rti/simulator.py` and `rti/schema.json` into that lakehouse's `Files/rti/` folder. No cloud resources are provisioned by this notebook.

Default: finite Delta microbatches, append-only events, matching existing dimensions required, existing RunId rejected. Eventhouse is an independent alternative, not a synchronized copy. See `rti/README.md` for table/mapping setup, secure Entra authentication, dimension loading, expected latency and Activator consent. Badge scans and device assignments are synthetic.

In [ ]:
# Delta uses the Fabric-provided PySpark/Delta runtime; no install needed.
# Eventhouse only: install rti/requirements-eventhouse.txt through a Fabric
# Environment, publish/attach it, and restart the session before continuing.
from pathlib import Path
import sys
module_folder = Path('/lakehouse/default/Files/rti')
assert (module_folder / 'simulator.py').is_file(), 'Attach default lakehouse and upload rti files first'
sys.path.insert(0, str(module_folder))
from simulator import Config, Simulator, DeltaSink, EventhouseSink, SOURCE_TABLES, UTC, run
import datetime as dt
import dataclasses
import os
import uuid


In [ ]:
MODE = 'delta'  # or 'eventhouse'; not automatically synchronized
RUN_ID = 'run-' + uuid.uuid4().hex  # set explicitly with START for repeatable offline replay
START = None  # None starts when the write cell runs; explicit aware datetime for replay
SEED = 20260915
BATCHES = 12  # finite, about one minute
INTERVAL_SECONDS = 5.0
SCANS_PER_MINUTE = 6.0  # per booth, multiplied by illustrative tier weights
CONFERENCE_ID = 0  # all conferences; use 1 for the outage walkthrough
OUTAGE_BOOTH_ID = 0  # disabled; choose a BoothId from preview below
OUTAGE_START_SECONDS = 600.0
OUTAGE_END_SECONDS = 1200.0
REALTIME = True  # False only for an offline replay, not live latency measurement


In [ ]:
# Demo-size model only: collect structural keys plus sponsor labels, not contact details.
columns = {
 'conference': ['ConferenceId'], 'sponsor': ['SponsorId', 'Name'],
 'conferencesponsor': ['ConferenceSponsorId', 'ConferenceId', 'SponsorId', 'Tier'],
 'user': ['UserId'], 'registration': ['ConferenceId', 'UserId'],
 'userlicence': ['UserId', 'Status']}
tables = {name: [row.asDict() for row in spark.read.table(f'`{name}`').select(*columns[name]).collect()]
          for name in SOURCE_TABLES}
config = Config(run_id=RUN_ID, start=START or dt.datetime.now(UTC), seed=SEED, batches=BATCHES,
 interval_seconds=INTERVAL_SECONDS, scans_per_minute=SCANS_PER_MINUTE,
 conference_id=CONFERENCE_ID, outage_booth_id=OUTAGE_BOOTH_ID,
 outage_start_seconds=OUTAGE_START_SECONDS, outage_end_seconds=OUTAGE_END_SECONDS)
sim = Simulator(tables, config)
print(sim.run_info())
display(spark.createDataFrame(sim.booths))


## Review before running
The next cell writes to the selected sink. For Eventhouse, explicitly configure `KUSTO_CLUSTER` and `KUSTO_DATABASE` in the runtime and use a supported Entra identity via `DefaultAzureCredential` (or pass an approved Azure TokenCredential to `EventhouseSink`). Do not paste tokens, client secrets, or connection strings into any cell or saved output. Fabric notebook identity support varies: if no approved credential is available, stop and configure one; there is no token workaround.

Run `setup.kql` manually and load both dimensions before Eventhouse mode. An Eventstream item by itself is not wired. This notebook does not create or activate Activator rules or share contacts. `IsQualified` = opted-in demo/meeting; `IsLicensedUser` is independent. Licensed-qualified = both flags.

In [ ]:
if MODE == 'delta':
    sink = DeltaSink(spark)
elif MODE == 'eventhouse':
    sink = EventhouseSink(os.environ['KUSTO_CLUSTER'], os.environ['KUSTO_DATABASE'])
else:
    raise ValueError('Supported notebook modes: delta, eventhouse')
if START is None:
    sim = Simulator(tables, dataclasses.replace(config, start=dt.datetime.now(UTC)))
total = run(sim, sink, realtime=REALTIME)
print(f'Finished {RUN_ID}: {total} events accepted by sink; verify query visibility separately.')
